# 🎓 Latih AI penguji VOIC (Colab gratis)

**Guru:** Kimi K3 — sudah membuat contoh percakapan ujian lewat `php artisan voic:teacher-generate` di laptop.
**Murid:** Qwen3-4B — dilatih di sini meniru gaya Kimi K3, lalu dijalankan **100% lokal di laptop lewat Ollama (CPU only)**.

Jalankan sel dari atas ke bawah (tombol ▶ di kiri setiap sel). Total waktu ± 30–60 menit.

| Langkah | Di mana |
|---|---|
| 1. Buat data dengan Kimi K3 | Laptop: `php artisan voic:teacher-generate` lalu `php artisan voic:export-training-data` |
| 2. Latih murid | **Notebook ini** |
| 3. Pasang di Ollama | Laptop: `ollama create voic-coach -f Modelfile` |

## Langkah 0 — Nyalakan GPU gratis
Menu **Runtime → Change runtime type → T4 GPU → Save**. Lalu jalankan sel di bawah: harus muncul tulisan **Tesla T4**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Langkah 1 — Pasang alat training (± 3 menit)

In [ ]:
%%capture
!pip install -q unsloth

## Langkah 2 — Masukkan 3 file dari laptop
File yang dibutuhkan (dari folder proyek VOIC):
- `ai_service/datasets/sft/voic_exam_sft.jsonl` (data latih)
- `ai_service/datasets/sft/voic_exam_eval.jsonl` (data uji)
- `ai_service/finetune_lora.py` (skrip training)

Atau cukup **satu file**: `ai_service/datasets/sft/voic_colab.zip` (berisi ketiganya).

Pilih **salah satu** cara, lalu jalankan sel di bawah. Sel ini otomatis mencari file di semua tempat ini:
1. **Panel Files (paling mudah):** klik ikon 📁 di kiri → seret file/zip ke panel itu → tunggu lingkaran upload di bawah selesai.
2. **Google Drive:** taruh file/zip di folder `VOIC` di Google Drive-mu (`MyDrive/VOIC`). Sel akan minta izin akses Drive.
3. **Tombol Choose Files:** muncul hanya kalau cara 1 & 2 tidak menemukan file. Kalau tombolnya tidak muncul/macet, pakai cara 1 atau 2 (tombol ini sering gagal di Brave/Edge atau kalau cookie pihak ketiga diblokir).

In [ ]:
import glob, os, shutil, zipfile

NEEDED = {
    "voic_exam_sft.jsonl": "datasets/sft/voic_exam_sft.jsonl",
    "voic_exam_eval.jsonl": "datasets/sft/voic_exam_eval.jsonl",
    "finetune_lora.py": "finetune_lora.py",
}
os.makedirs("datasets/sft", exist_ok=True)

def missing():
    return [name for name, target in NEEDED.items() if not os.path.exists(target)]

def collect(folder):
    """Salin file yang dibutuhkan dari folder (termasuk isi zip) ke tempat yang dipakai training."""
    # /content hanya level atas (jangan menjelajah seluruh Drive yang ter-mount di /content/drive).
    deep = folder != "/content"
    pattern = lambda base, name: glob.glob(os.path.join(base, "**", name) if deep else os.path.join(base, name), recursive=deep)
    for archive in pattern(folder, "*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall("/content/_voic_zip")
    for source_dir in (folder, "/content/_voic_zip"):
        deep = deep or source_dir == "/content/_voic_zip"
        for name, target in NEEDED.items():
            if os.path.exists(target):
                continue
            # Colab menamai ulang file yang diunggah dua kali: "voic_exam_sft (1).jsonl".
            stem, ext = os.path.splitext(name)
            hits = sorted(pattern(source_dir, name) + pattern(source_dir, f"{stem} (*){ext}"))
            if hits:
                shutil.copy(hits[-1], target)

# 1) Panel Files (file diseret ke /content)
collect("/content")

# 2) Google Drive: MyDrive/VOIC
if missing():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir("/content/drive/MyDrive/VOIC"):
            collect("/content/drive/MyDrive/VOIC")
        else:
            print("Folder MyDrive/VOIC belum ada di Google Drive.")
    except Exception as exc:
        print("Google Drive tidak bisa dipakai:", exc)

# 3) Tombol Choose Files (cadangan terakhir)
if missing():
    print("Masih kurang:", missing(), "→ pilih filenya lewat tombol di bawah.")
    try:
        from google.colab import files
        files.upload()
        collect("/content")
    except Exception as exc:
        print("Tombol upload gagal:", exc, "→ pakai panel Files (ikon 📁) atau Google Drive.")

for name, target in NEEDED.items():
    if os.path.exists(target):
        lines = sum(1 for _ in open(target, encoding="utf-8")) if target.endswith(".jsonl") else None
        print("✓", target, f"({lines} contoh)" if lines is not None else "")
    else:
        print("✗ belum ada:", name)
assert not missing(), "Masukkan file yang kurang (lihat cara 1–3 di atas), lalu jalankan sel ini lagi."

## Langkah 3 — Latih murid (± 20–40 menit)
- Data < 200 contoh: biarkan `--epochs 2` agar tidak "menghafal".
- Data > 500 contoh: boleh `--epochs 3`.

Di akhir, skrip otomatis mengubah model menjadi file **GGUF** (format Ollama) + **Modelfile**.

In [ ]:
!python finetune_lora.py \
    --train datasets/sft/voic_exam_sft.jsonl \
    --eval datasets/sft/voic_exam_eval.jsonl \
    --base unsloth/Qwen3-4B-Instruct-2507 \
    --epochs 2 \
    --output outputs/voic-coach

## Langkah 4 — Simpan hasil ke Google Drive (disarankan)
File GGUF ± 2,5 GB. Menyimpan ke Drive lebih aman daripada unduh langsung (unduhan besar di browser sering putus).
Setelah selesai, buka Google Drive → folder **VOIC-model**, unduh **file .gguf** dan **Modelfile** ke satu folder yang sama di laptop.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

target = "/content/drive/MyDrive/VOIC-model"
os.makedirs(target, exist_ok=True)
for name in os.listdir("outputs/voic-coach"):
    if name.endswith(".gguf") or name == "Modelfile":
        shutil.copy(os.path.join("outputs/voic-coach", name), target)
        print("✓ disalin:", name)

## Langkah 5 — Pasang di laptop (tanpa internet, CPU only)
Buka Command Prompt di folder tempat file **.gguf** dan **Modelfile** berada, lalu:

```bash
ollama create voic-coach -f Modelfile
```

Pastikan di file `.env` Laravel baris `VOIC_AI_MODEL=` **kosong** (agar memakai model lokal `voic-coach`), lalu nyalakan ulang `jalankan_voic.bat`.

Bandingkan hasilnya (bukti untuk dosen):

```bash
cd ai_service
python eval_dialogue.py --models voic-coach qwen3:4b
```